### Import packages

In [ ]:
from enum import Enum
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

### Define parameters

In [ ]:
N = 24				# number of time steps
I = 2				# number of containers
DELTA_t = 1.0		# hr

ALPHA = 2.61e-5		# MWh capacity lost per MWh throughput
BETA = 2.5			# °C / MWh throughput
GAMMA = 3.0			# °C / hr
ETA = 0.968			# dimensionless
KAPPA = 0.04		# / °C
RHO = 19500			# $ / MWh capacity lost (opportunity)
SIGMA = 200000		# $ / MWh capacity lost (replacement)
A = 0.02			# dimensionless
B = 0.02			# dimensionless

T_MIN = -30			# °C
T_REF = 25			# °C
T_MAX = 50			# °C

SOC_MIN = 0.05		# dimensionless
SOC_MAX = 0.95		# dimensionless
Q_NEW = 3.916		# MWh, new container capacity
P_MAX = 0.979		# MW, hard limit

# Day starting values for both containers (arbitrarily chosen)
SOH_0 = np.array([0.98, 0.82])	# dimensionless
SOC_0 = np.array([0.8, 0.4])	# dimensionless

# Day ending values
SOC_N = SOC_0

In [ ]:
class Season(Enum):
	WINTER = 'jan'
	SPRING = 'apr'
	SUMMER = 'jul'
	AUTUMN = 'oct'

In [ ]:
season = Season.WINTER

### Generate market price trajectory

In [ ]:
expected_prices_df = pd.read_csv("./data/raw/DAM/seasonal_stats.csv")
expected_prices_df.head()

In [ ]:
LAMBDA = expected_prices_df[f'{season.value}_mean'].to_numpy().reshape(-1, 1)

### Define state & decision variables

In [ ]:
# states: k = 0 -> N, i = 1 -> 2
E = cp.Variable((N + 1, I), nonneg=True)
Q = cp.Variable((N + 1, I), nonneg=True)
T = cp.Variable((N + 1, I), nonneg=True)

# decisions: k = 0 -> N-1, i = 1 -> 2
c = cp.Variable((N, I), nonneg=True)
d = cp.Variable((N, I), nonneg=True)
u = cp.Variable((N, I), nonneg=True)

### Define constraints

In [ ]:
phi = cp.Parameter((N,I), nonneg=True)
delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t

constraints = [
	# Initial conditions
	Q[0, :] == Q_NEW * SOH_0,
	E[0, :] == cp.multiply(Q[0, :], SOC_0),
	T[0, :] == T_REF,
	# Dynamics - see below for capacity dynamics
	Q[1:, :] == Q[:-1, :] - delta_Q,
	E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
	T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
	# Boundary conditions
	# Q >= 0, # variable created as non-neg=True
	E >= Q * SOC_MIN,
	E <= Q * SOC_MAX,
	T >= T_MIN,
	T <= T_MAX,
	# c >= 0, # variable created as non-neg=True
	c <= P_MAX,
	# d >= 0, # variable created as non-neg=True
	d <= P_MAX,
	# u >= 0, # variable created as non-neg=True
	u <= 1,
	# Terminal conditions
	E[N, :] >= cp.multiply(Q[N, :], SOC_N)
]

### Define objective function and problem

In [ ]:
arbitrage_cost = cp.sum(cp.multiply(LAMBDA, c - d) * DELTA_t)
operating_cost = cp.sum(cp.multiply(LAMBDA, A + B * u) * P_MAX * DELTA_t)
opportunity_plus_replacement_cost = cp.sum((RHO + SIGMA) * delta_Q)
objective = cp.Minimize(arbitrage_cost + opportunity_plus_replacement_cost + operating_cost)
problem = cp.Problem(objective, constraints)

### Successive convex programming

Unfortunately, our temperature-dependent state transition function for capacity is non-convex. 

The following workaround solves the problem iteratively instead:
1. Temperature remains a regular state variable. But _only when calculating the temperature degradation factor_ $\phi(k)$, we initially assume the temperature is $T_\text{ref}$ the whole time.
2. We solve the problem. Temperature changes based on our optimized decisions, but the _degradation_ is as if it stayed static.
3. We plug in our solved values for temperature as a _static_ guess in order to calculate another iteration of $\phi(k)$, and repeat.
4. This converges in just three loops (I tried 10 initially). That is, the temperature state trajectory stops changing with each iteration.

Essentially, while temperature is a state variable, we use the previous iteration's trajectory for temperature as a fixed parameter to calculate $\phi(k)$, making the problem convex.

In [ ]:
T_est = np.full((N, I), T_REF)

for iteration in range(4):
	print(f"Iteration {iteration} starting...")

	phi.value = 1 + KAPPA * np.maximum(0, T_est - T_REF)

	problem.solve(
		solver=cp.ECOS,
		verbose=True
	)

	if problem.status == "optimal":
		T_est = T.value[:-1, :]
		print(f"Iteration {iteration}: Total Cost = {problem.value:.2f}")
	else:
		print("Solver failed to find an optimal solution.")
		break


### Plot

In [ ]:
fig, axes = plt.subplots(5, figsize=(8,20))
K = np.array(range(N+1))

axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Stored Energy (MWh)', fontweight='bold')
axes[0].plot(K, E.value[:,0], color='blue', linewidth=2, label='Container 1', marker='+')
axes[0].plot(K, E.value[:,1], color='red', linewidth=2, label='Container 2', marker='x')
axes[0].tick_params(axis='y')
axes[0].set_ylim(0, Q_NEW + 1) # Slightly above max capacity
axes[0].set_title('BESS Energy Dispatch vs. Market Price')
axes[0].legend()

ax2 = axes[0].twinx()
ax2.set_ylabel('RTM Price ($/MWh)', fontweight='bold')
ax2.step(K[:-1], LAMBDA, where='post', alpha=0.6, label='Price')
ax2.tick_params(axis='y')

axes[1].plot(K, Q.value[:,0] - Q.value[0,0], label="Container 1", color='blue', marker='+')
axes[1].plot(K, Q.value[:,1] - Q.value[0,1], label="Container 2", color='red', marker='x')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Change in Capacity (MWh)', fontweight='bold')
axes[1].set_title("Change in Container Capacity")
axes[1].legend()

axes[2].plot(K, T.value[:,0], label="Container 1", color='blue', marker='+')
axes[2].plot(K, T.value[:,1], label="Container 2", color='red', marker='x')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Temperature', fontweight='bold')
axes[2].set_title("Container Temperature")
axes[2].legend()

axes[3].plot(K[:-1], c.value[:,0], marker='^', color='blue', label="C1 Charging")
axes[3].plot(K[:-1], -d.value[:,0], marker='v', color='blue', linestyle='--', label="C1 Charging")
axes[3].plot(K[:-1], c.value[:,1], marker='^', color='red', label="C2 Discharging")
axes[3].plot(K[:-1], -d.value[:,1], marker='v', color='red', linestyle='--', label="C2 Discharging")
axes[3].set_xlabel("Hour of Day")
axes[3].set_ylabel("Power setpoint (MW)")
axes[3].legend()
axes[3].set_title("Power setpoints")

axes[4].plot(K[:-1], u.value[:,0], label="Container 1", color='blue', marker='+')
axes[4].plot(K[:-1], u.value[:,1], label="Container 2", color='red', marker='x')
axes[4].set_xlabel('Hour of Day')
axes[4].set_ylabel('Cooling effort [0,1]', fontweight='bold')
axes[4].set_title("Cooling Effort")
axes[4].legend()

plt.grid(axis='x', linestyle='--')
plt.tight_layout()
plt.show()
